## Complexity Grouping

The dataset contains examples requiring different numbers of expected tool calls.

Direct stratification using the exact number of tool calls is not possible because some categories contain very few examples.

Therefore, examples are grouped into broader complexity categories:

- `1_call`
- `2_calls`
- `3_calls`
- `4_plus_calls`

These groups will be used for stratified splitting.

In [ ]:
def create_complexity_group(example):

    answers = json.loads(example["answers"])
    num_calls = len(answers)

    if num_calls == 1:
        group = "1_call"

    elif num_calls == 2:
        group = "2_calls"

    elif num_calls == 3:
        group = "3_calls"

    else:
        group = "4_plus_calls"

    return {
        "complexity_group": group
    }


dataset_grouped = dataset["train"].map(
    create_complexity_group
)

print("FULL DATASET DISTRIBUTION:")

print(
    Counter(
        dataset_grouped["complexity_group"]
    )
)

# Data Preparation and Dataset Splitting

## Objective

This notebook prepares the xLAM function-calling dataset for model training.

The objectives are to:

- Create complexity groups based on the number of expected tool calls
- Perform a grouped stratified dataset split
- Create training, validation, and test datasets
- Verify dataset sizes and complexity distributions across splits

The final dataset split is:

- Training: 80%
- Validation: 10%
- Test: 10%

In [ ]:
import json
from collections import Counter
from sklearn.model_selection import train_test_split

In [ ]:
# Create indices for all dataset examples
indices = list(range(len(dataset_grouped)))

#Complexity groups used as stratification labels
stratify_labels = dataset_grouped["complexity_group"]
train_indices, temp_indices = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    stratify=stratify_labels
)

# Get complexity labels corresponding to the temporary indices
temp_labels = [
    stratify_labels[i]
    for i in temp_indices
]

# Split temporary dataset into validation and test sets
validation_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

train_dataset = dataset_grouped.select(train_indices)

validation_dataset = dataset_grouped.select(validation_indices)

test_dataset = dataset_grouped.select(test_indices)


# Creating Representative Datasets

To make experimentation more computationally efficient, representative subsets are created from the stratified train, validation, and test datasets. The subsets preserve the distribution of tool-call complexity.

In [ ]:
from datasets import concatenate_datasets

REPRESENTATIVE_TRAIN_SIZE = 4000
REPRESENTATIVE_VALIDATION_SIZE = 1000
REPRESENTATIVE_TEST_SIZE = 1000

RANDOM_SEED = 42


def create_representative_dataset(dataset, target_size):
    """Create a complexity-stratified representative subset."""

    complexity_groups = {
        "1_call": dataset.filter(
            lambda x: x["complexity"] == "1_call"
        ),
        "2_calls": dataset.filter(
            lambda x: x["complexity"] == "2_calls"
        ),
        "3_calls": dataset.filter(
            lambda x: x["complexity"] == "3_calls"
        ),
        "4_plus_calls": dataset.filter(
            lambda x: x["complexity"] == "4_plus_calls"
        ),
    }

    total_size = len(dataset)
    representative_parts = []

    for complexity, group in complexity_groups.items():

        proportion = len(group) / total_size
        target_group_size = round(target_size * proportion)

        target_group_size = min(
            target_group_size,
            len(group)
        )

        if target_group_size > 0:
            representative_parts.append(
                group
                .shuffle(seed=RANDOM_SEED)
                .select(range(target_group_size))
            )

    representative_dataset = concatenate_datasets(
        representative_parts
    )

    return representative_dataset.shuffle(
        seed=RANDOM_SEED
    )


representative_train_dataset = create_representative_dataset(
    train_dataset,
    REPRESENTATIVE_TRAIN_SIZE
)

representative_validation_dataset = create_representative_dataset(
    validation_dataset,
    REPRESENTATIVE_VALIDATION_SIZE
)

representative_test_dataset = create_representative_dataset(
    test_dataset,
    REPRESENTATIVE_TEST_SIZE
)

print("Representative datasets created.")
print(f"Train: {len(representative_train_dataset)}")
print(f"Validation: {len(representative_validation_dataset)}")
print(f"Test: {len(representative_test_dataset)}")

# Save Representative Datasets

The representative datasets are saved to Google Drive so that the same data can be reused consistently across the training and evaluation notebooks without recreating the sampling process.

In [ ]:

from pathlib import Path

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/tool_calling_project"
)

REPRESENTATIVE_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "representative_data"
)

REPRESENTATIVE_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)


representative_train_dataset.save_to_disk(
    str(REPRESENTATIVE_DATA_DIR / "train")
)

representative_validation_dataset.save_to_disk(
    str(REPRESENTATIVE_DATA_DIR / "validation")
)

representative_test_dataset.save_to_disk(
    str(REPRESENTATIVE_DATA_DIR / "test")
)

print("Representative datasets saved successfully.")
print(f"Location: {REPRESENTATIVE_DATA_DIR}")